# Graph Animation Examples

This notebook shows several schedules directly, without helper builder functions. Each example is constructed in its own code cell.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyleTransition,
    DrawTransition,
    EraseTransition,
    FillBetweenArea,
    FillBetweenTransition,
    FillStyleTransition,
    JitterTransition,
    MoveFillBetweenTransition,
    MoveTransition,
    ParallelTransition,
    Schedule,
    StressTransition,
)


In [ ]:
# Example 1: draw two curves concurrently
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))
y_arch = np.clip(0.2 + 0.7 * (1.0 - np.square(2.0 * x - 1.0)), 0.0, 1.0)

schedule = Schedule().add(
    ParallelTransition(
        (
            DrawTransition(Curve("wave_a", x, y_wave, color="#0f766e", linewidth=3.0)),
            DrawTransition(
                Curve(
                    "arch_a",
                    x,
                    y_arch,
                    color="#2563eb",
                    linewidth=2.5,
                    linestyle="--",
                )
            ),
        )
    ),
    duration=2.0,
)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
anim = schedule.build_animation(ax=ax, fps=24, title="Parallel Draws")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 2: draw a curve and its fill_between at the same time
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))

schedule = Schedule()
schedule.add(
    ParallelTransition(
        (
            DrawTransition(Curve("wave_b", x, y_wave, color="#7c3aed", linewidth=3.0)),
            FillBetweenTransition(
                FillBetweenArea(
                    "wave_b_fill",
                    x,
                    y_wave,
                    0.0,
                    color="#c4b5fd",
                    alpha=0.25,
                    linewidth=1.0,
                )
            ),
        )
    ),
    duration=2.0,
)
schedule.add(StressTransition("wave_b", color="#a78bfa", max_alpha=0.25), duration=0.8)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
anim = schedule.build_animation(ax=ax, fps=24, title="Draw And Fill Together")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 3: sudden plot and sudden style change with zero-duration transitions
x = np.linspace(0.0, 1.0, 250)
y_arch = np.clip(0.2 + 0.7 * (1.0 - np.square(2.0 * x - 1.0)), 0.0, 1.0)

schedule = Schedule()
schedule.add(
    DrawTransition(
        Curve("snap", x, y_arch, color="#0ea5e9", linewidth=2.0),
        show_pointer=False,
    ),
    duration=0.0,
)
schedule.add(
    CurveStyleTransition("snap", color="#dc2626", linestyle="--", linewidth=3.0),
    duration=0.0,
)
schedule.add(StressTransition("snap", color="#fb923c", max_alpha=0.25), duration=0.8)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
anim = schedule.build_animation(ax=ax, fps=24, title="Sudden Plot And Style Change")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 4: slowly change alpha and color
x = np.linspace(0.0, 1.0, 250)
y_ramp = 0.12 + 0.72 * np.power(x, 1.1)

schedule = Schedule()
schedule.add(
    DrawTransition(
        Curve("fade", x, y_ramp, color="#94a3b8", alpha=0.0, linewidth=3.0),
        show_pointer=False,
    ),
    duration=0.0,
)
schedule.add(CurveStyleTransition("fade", color="#0f766e", alpha=1.0), duration=1.5)
schedule.add(CurveStyleTransition("fade", color="#f59e0b", alpha=0.15), duration=1.0)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
anim = schedule.build_animation(ax=ax, fps=24, title="Fade And Color Shift")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 5: concurrent draw/fill, then stress, jitter, and move both together
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))
y_square = np.square(y_wave)

schedule = Schedule()
schedule.add(
    ParallelTransition(
        (
            DrawTransition(Curve("story_wave", x, y_wave, color="#0f766e", linewidth=3.0)),
            FillBetweenTransition(
                FillBetweenArea(
                    "story_fill",
                    x,
                    y_wave,
                    0.0,
                    color="#99f6e4",
                    alpha=0.2,
                )
            ),
        )
    ),
    duration=2.0,
)
schedule.add(StressTransition("story_wave", color="#14b8a6", max_alpha=0.3), duration=0.8)
schedule.add(JitterTransition("story_wave", y_amplitude=0.03, cycles=12.0, seed=7), duration=0.8)
schedule.add(
    ParallelTransition(
        (
            MoveTransition("story_wave", x_prime=x, y_prime=y_square),
            MoveFillBetweenTransition(
                "story_fill",
                x_prime=x,
                y1_prime=y_square,
                y2_prime=0.0,
            ),
            CurveStyleTransition("story_wave", color="#7c3aed"),
            FillStyleTransition("story_fill", color="#c4b5fd", alpha=0.12),
        )
    ),
    duration=2.0,
)
schedule.add(EraseTransition("story_wave"), duration=1.2)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
anim = schedule.build_animation(ax=ax, fps=24, title="Parallel Move With Effects")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())
